# Phase 3: Backtest and baselines

**Goal:** a scoring function that tells us, in seconds, how many hits any ranking would get. Every later decision (features, models, Granite text scores) is judged with it.

**Why a backtest?** The real test is graded on KEV additions through November 4, 2026, which we can't see. So we replay the same test on 2025, where we already know the answers:

1. Take every vulnerability published January-September 2025 that was **not** on KEV by the end of its publish month (the same rule the 2026 scoreboard uses).
2. Give it only the signals known at the end of that month (EPSS, exploit code flags).
3. Label it `y = 1` if CISA added it to KEV later, by December 31, 2025.
4. Score a ranking by how many `y = 1` rows land in its top **243**, the same 0.69% cut as 500 of 72,670.

If the three standard rankings score about the same here as on the official benchmark, the backtest is a fair stand-in.

**Outcome of this notebook:**

| | Rows | Positives | Top k |
|---|---|---|---|
| Validation (2025) | 35,281 | 52 | 243 |
| Training (2024) | 28,839 | 33 | 198 |

| Baseline | hits@243 | Official 2026 top 500 |
|---|---|---|
| CVSS first | 5 | 7 |
| EPSS first | 8 | 6 |
| Exploit code then EPSS | **14** | 8 |

**The number every later idea must beat is 14**, by more than the bootstrap noise.

**Before running:** from the repo root, run `python3 scripts/fetch_data.py` once to download `data/`, then `pip install duckdb pandas numpy`.

## 1. Load the data

We use **DuckDB**, a database that runs inside Python and queries CSV files directly, so there's nothing to set up. Joins over millions of rows take seconds, and only the small results come back into pandas.

The CSVs contain descriptions with commas and quotes, so we set `quote` and `escape` to `"` as the data dictionary says.

In [1]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

# Works whether the notebook runs from notebooks/ or the repo root.
DATA = next(p for p in [Path("../data"), Path("data")] if p.exists()).resolve()
con = duckdb.connect()
for t in ["vulnerabilities", "known_exploited", "exploit_signals_2024",
          "exploit_signals_2025", "scoreboard_2026"]:
    con.sql(f'''CREATE VIEW "{t}" AS
                SELECT * FROM read_csv('{DATA}/{t}.csv', quote='"', escape='"', sample_size=-1)''')

**Check row counts against the README.** A mismatch would mean rows broke apart on a quoting problem.

In [2]:
expected = {"vulnerabilities": 401_082, "known_exploited": 1_484, "scoreboard_2026": 72_670,
            "exploit_signals_2024": None, "exploit_signals_2025": None}
counts = {t: con.sql(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0] for t in expected}
for t, n in counts.items():
    print(f"{t:22} {n:>10,}", "" if expected[t] is None else ("OK" if n == expected[t] else "MISMATCH"))
signals_total = counts["exploit_signals_2024"] + counts["exploit_signals_2025"]
print(f"{'signals 2024 + 2025':22} {signals_total:>10,}", "OK" if signals_total == 6_463_734 else "MISMATCH")
assert all(expected[t] in (None, n) for t, n in counts.items()) and signals_total == 6_463_734

vulnerabilities           401,082 OK
known_exploited             1,484 OK
scoreboard_2026            72,670 OK
exploit_signals_2024    3,023,284 
exploit_signals_2025    3,440,450 
signals 2024 + 2025     6,463,734 OK


### Optional: cross-check against Tiger Data

Our Tiger Cloud database holds one table, `exploit_signals`: the 2024-2025 monthly signals (`source = 'signals'`) and the 2026 scoreboard values (`source = 'scoreboard'`). It does **not** hold the vulnerabilities or the KEV list, so the backtest reads those from the CSVs either way.

This cell checks that the database matches the CSVs, so the dashboard and the model see the same numbers. It reads `DATABASE_URL` from the environment, or else from `apps/web/.env.local` (the same file the web app uses, which git ignores). It's skipped if neither has it. Don't paste the URL into the notebook: it contains the password and would end up in git.

In [3]:
import os
url = os.environ.get("DATABASE_URL")
env_file = DATA.parent / "apps/web/.env.local"
if not url and env_file.exists():
    for line in env_file.read_text().splitlines():
        if line.strip().startswith("DATABASE_URL="):
            url = line.split("=", 1)[1].strip().strip('"').strip("'")
if not url:
    print("DATABASE_URL not set; skipping the Tiger Data check.")
else:
    con.sql("INSTALL postgres; LOAD postgres;")
    con.sql(f"ATTACH '{url}' AS tiger (TYPE postgres, READ_ONLY)")
    tiger = con.sql("SELECT source, COUNT(*) AS n FROM tiger.public.exploit_signals GROUP BY source").df()
    print(tiger)
    n = dict(zip(tiger.source, tiger.n))
    assert n.get("signals") == signals_total and n.get("scoreboard") == counts["scoreboard_2026"], \
        "Tiger Data does not match the CSVs"
    print("Tiger Data matches the CSVs.")

       source        n
0     signals  6463734
1  scoreboard    72670
Tiger Data matches the CSVs.


## 2. Build the evaluation sets

One function builds both sets, so training and validation follow exactly the same rules.

**Rows:** published between `start` and `end`, and not on KEV by the end of the publish month. On the scoreboard, a vulnerability already on KEV is excluded, because predicting it would be trivial.

**Label:** `y = 1` if added to KEV *after* the publish month and on or before `deadline`.

**Signals:** EPSS and the four exploit code flags from the signals file **for the publish month**, which is what was known at the time. Vulnerabilities with no signals that month get blank EPSS and 0 flags, like the scoreboard.

**Columns kept:** only the Safe and Careful columns from the team plan's leakage table. These are left out on purpose, because NIST fills them in *after* attacks: `nvd_status`, `n_affected_cpes`, `n_references`, `cvss3_source` and `last_modified`.

Two details:
- `published` sets the period, not the year in the CVE ID. CVE-2025-xxxx can be published in 2026.
- The 62 vulnerabilities whose KEV date is *before* their publish date count as "already on KEV", so they're excluded rather than counted as positives.

In [4]:
FLAGS = ["exploitdb", "metasploit", "nuclei", "poc_github"]

def eval_set(start, end, deadline, signals):
    flags = ", ".join(f"COALESCE(CAST(s.{f} AS INT), 0) AS {f}" for f in FLAGS)
    df = con.sql(f'''
        WITH v AS (SELECT *, CAST(published AS DATE) AS pub FROM vulnerabilities),
             k AS (SELECT cve_id, MIN(CAST(date_added AS DATE)) AS date_added
                   FROM known_exploited GROUP BY cve_id)
        SELECT v.cve_id, v.pub AS published, strftime(v.pub, '%Y-%m') AS snap,
               v.assigner, v.description, CAST(v.cvss3_score AS DOUBLE) AS cvss3_score,
               v.attack_vector, v.attack_complexity, v.privileges_required,
               v.user_interaction, v.scope, v.cwe_ids,
               CAST(s.epss AS DOUBLE) AS epss, {flags},
               k.date_added AS kev_date_added,
               CAST(k.date_added IS NOT NULL AND k.date_added <= DATE '{deadline}' AS INT) AS y
        FROM v
        LEFT JOIN k USING (cve_id)
        LEFT JOIN {signals} s ON s.cve_id = v.cve_id AND s.month = strftime(v.pub, '%Y-%m')
        WHERE v.pub BETWEEN DATE '{start}' AND DATE '{end}'
          AND (k.date_added IS NULL OR k.date_added > last_day(v.pub))
        ORDER BY v.cve_id  -- fixed row order: tie-breaks and bootstraps depend on it
    ''').df()
    df["any_exploit"] = (df[FLAGS].sum(axis=1) > 0).astype(int)
    # Months from the snapshot to the label deadline: a January CVE has more time to be added.
    df["months_left"] = (pd.Timestamp(deadline).to_period("M")
                         - pd.to_datetime(df["snap"]).dt.to_period("M")).apply(lambda d: d.n)
    return df

### Validation set: 2025

Published January-September 2025, labels through December 31, 2025. This mirrors the 2026 benchmark: published January-September, labels through early November.

In [5]:
val = eval_set("2025-01-01", "2025-09-30", "2025-12-31", "exploit_signals_2025")
print(f"{len(val):,} rows, {val.y.sum()} positives ({val.y.mean():.2%})")
assert len(val) == 35_281 and val.y.sum() == 52

35,281 rows, 52 positives (0.15%)


### Training set: 2024

The same rules shifted one year back: published January-September 2024, labels through December 31, 2024. Keeping the window the same length means a model learns the same thing it's tested on, "added within the rest of this calendar year."

There are only 33 positives. That's why Phase 5 starts with simple models, and why Track B (text only) trains on 2018-2024 descriptions, which don't need monthly signals.

In [6]:
train = eval_set("2024-01-01", "2024-09-30", "2024-12-31", "exploit_signals_2024")
print(f"{len(train):,} rows, {train.y.sum()} positives ({train.y.mean():.2%})")

28,839 rows, 33 positives (0.11%)


### Sanity checks

**Leakage:** no banned column is present, and every positive was added to KEV after its publish month and by the deadline.

**Signal coverage:** the share of rows that have an EPSS score at the end of their publish month.

In [7]:
banned = {"nvd_status", "n_affected_cpes", "n_references", "cvss3_source", "last_modified"}
for name, d, deadline in [("val", val, "2025-12-31"), ("train", train, "2024-12-31")]:
    assert not banned & set(d.columns)
    pos = d[d.y == 1]
    month_end = pd.to_datetime(pos.published) + pd.offsets.MonthEnd(0)
    assert (pd.to_datetime(pos.kev_date_added) > month_end).all()
    assert (pd.to_datetime(pos.kev_date_added) <= pd.Timestamp(deadline)).all()
print("Leakage checks passed.\n")

summary = val.groupby("snap").agg(rows=("y", "size"), positives=("y", "sum"),
                                  epss_coverage=("epss", lambda s: s.notna().mean()),
                                  months_left=("months_left", "first"))
summary.round(2)

Leakage checks passed.



,rows,positives,epss_coverage,months_left
snap,,,,
2025-01,4267,7,0.55,11
2025-02,3664,6,0.77,10
2025-03,3982,7,0.92,9
2025-04,4024,5,0.98,8
2025-05,3968,6,1.00,7
2025-06,3674,8,0.99,6
2025-07,3768,2,0.98,5
2025-08,3621,5,0.99,4
2025-09,4313,6,0.98,3


**What this shows:** positives are spread across all nine months (2 to 8 each), not bunched in one. EPSS coverage is low early in the year: only 55% of January rows and 77% of February rows have a score, against 92-100% from March on. (The data dictionary also notes that the February 2025 signals file is missing about 14,500 vulnerabilities.) Rows without a score are ranked last by EPSS-based rankings, so early-2025 vulnerabilities are under-ranked. Phase 4 should add a "signal missing" flag.

`months_left` falls from 11 to 3: a September vulnerability has much less time to be added before the deadline. That's a feature for Phase 4, not a bug.

## 3. The scoring function

`hits_at_k(df, score)` sorts rows best-first and counts positives in the top k.

- **k scales with the set's size**, so the cut matches the benchmark's 500 of 72,670 (0.69%): 243 for validation and 198 for training.
- **`score`** can be a column name, a list of columns (sort by the first, break ties with the next, as in "exploit code then EPSS"), or an array of model scores.
- **Blank scores go last.** A vulnerability with no EPSS isn't treated as high risk.
- **Ties are broken by a fixed random order.** Many vulnerabilities share a CVSS of 9.8, so which ones make the cut is otherwise arbitrary. A fixed `seed` makes every run give the same answer. That only holds if the rows are always in the same order, which is why `eval_set` sorts by `cve_id`: databases don't promise to return rows in a stable order.

In [8]:
BENCH_K, BENCH_N = 500, 72_670

def default_k(n):
    return int(round(n * BENCH_K / BENCH_N))

def rank_order(df, score, seed=0):
    # Row positions, best first.
    cols = [score] if isinstance(score, str) else score
    if isinstance(cols, list) and all(isinstance(c, str) for c in cols):
        keys = [df[c].to_numpy(dtype=float) for c in cols]
    else:
        keys = [np.asarray(score, dtype=float)]
    keys = [np.where(np.isnan(k), -np.inf, k) for k in keys]
    tie = np.random.default_rng(seed).permutation(len(df))
    # np.lexsort sorts ascending by the LAST key first, so negate and reverse.
    return np.lexsort([tie] + [-k for k in reversed(keys)])

def hits_at_k(df, score, k=None, seed=0):
    k = k or default_k(len(df))
    return int(df["y"].to_numpy()[rank_order(df, score, seed)[:k]].sum())

print("k for validation:", default_k(len(val)), "| k for training:", default_k(len(train)))

k for validation: 243 | k for training: 198


## 4. Reproduce the baselines

These are the three standard rankings from the README. If our backtest gives about the same numbers as the official benchmark, it's a fair stand-in for 2026.

In [9]:
BASELINES = {
    "CVSS first": "cvss3_score",
    "EPSS first": "epss",
    "Exploit code then EPSS": ["any_exploit", "epss"],
}
OFFICIAL = {"CVSS first": 7, "EPSS first": 6, "Exploit code then EPSS": 8}
pd.DataFrame([{"ranking": n, "val 2025 hits@243": hits_at_k(val, s),
               "train 2024 hits@198": hits_at_k(train, s),
               "official 2026 hits@500": OFFICIAL[n]} for n, s in BASELINES.items()])

,ranking,val 2025 hits@243,train 2024 hits@198,official 2026 hits@500
0,CVSS first,5,0,7
1,EPSS first,8,4,6
2,Exploit code then EPSS,14,7,8


**Reading this:** the winner matches the official benchmark: exploit code then EPSS is best in both. CVSS and EPSS swap places, though (official: CVSS 7, EPSS 6; here: CVSS 5, EPSS 8). Section 5 shows that gap is within noise here too, so the honest summary is "exploit code then EPSS wins; CVSS and EPSS can't be told apart." The exact numbers also differ (14 here vs. 8 officially for the best one), which is expected with so few positives: 52 here, and an unknown number in 2026. Use backtest scores to *compare* rankings, not to forecast the 2026 count.

On the 2024 training set, CVSS first finds **0** of 33, even though the positives' median CVSS is 9.1. Thousands of vulnerabilities score 9.8, so a high score barely narrows the list.

The takeaway for the write-up: **severity alone finds about 1 in 10 of the vulnerabilities later exploited.** Even the best standard ranking finds only about a quarter of them.

## 5. How much is noise? Bootstrap ranges

With 52 positives, one or two extra hits can be luck. A **bootstrap** estimates how much the score would move on a slightly different sample of vulnerabilities:

1. Draw 35,281 rows from the validation set *with replacement*, so some rows appear twice and some not at all.
2. Recount hits@243 on that sample.
3. Repeat 1,000 times. The middle 90% of the results is the range.

*Example:* if our model scores 20 with a range of 14-26 and the baseline is 14, the gain is promising but not certain.

In [10]:
def _ranks(df, score, seed):
    r = np.empty(len(df), dtype=np.int64)
    r[rank_order(df, score, seed)] = np.arange(len(df))
    return r

def _boot_hits(ranks, y, k, idx):
    # For each resample, take the k best-ranked sampled rows and sum their labels.
    return np.array([y[i][np.argpartition(ranks[i], k)[:k]].sum() for i in idx])

def bootstrap(df, score, k=None, n_boot=1000, ci=0.90, seed=0):
    k = k or default_k(len(df))
    idx = np.random.default_rng(seed).integers(0, len(df), size=(n_boot, len(df)))
    h = _boot_hits(_ranks(df, score, seed), df["y"].to_numpy(), k, idx)
    a = (1 - ci) / 2
    return int(np.quantile(h, a)), int(np.quantile(h, 1 - a))

baselines = pd.DataFrame([{"ranking": n, "hits@243": hits_at_k(val, s),
                           "90% range": "%d-%d" % bootstrap(val, s)} for n, s in BASELINES.items()])
baselines

,ranking,hits@243,90% range
0,CVSS first,5,2-9
1,EPSS first,8,4-13
2,Exploit code then EPSS,14,8-21


The ranges are wide and overlap. That's the honest picture with 52 positives, and the reason we compare models with the **paired** test below rather than by eyeballing two ranges.

### Comparing two rankings: the paired bootstrap

Two rankings tested on the *same* resamples share most of their luck, so the **difference** between them is much less noisy than either score alone. `compare(val, A, B)` reports:

- `diff`: hits of A minus hits of B on the real data
- `range`: the 90% range of that difference over 1,000 resamples. **If it doesn't include 0, A's lead is probably real.**
- `p_a_better`: the share of resamples where A wins outright

This is the function to use in Phases 5 and "Compare and combine".

In [11]:
def compare(df, score_a, score_b, k=None, n_boot=1000, ci=0.90, seed=0):
    k = k or default_k(len(df))
    idx = np.random.default_rng(seed).integers(0, len(df), size=(n_boot, len(df)))
    y = df["y"].to_numpy()
    d = (_boot_hits(_ranks(df, score_a, seed), y, k, idx)
         - _boot_hits(_ranks(df, score_b, seed), y, k, idx))
    a = (1 - ci) / 2
    return {"diff": hits_at_k(df, score_a, k, seed) - hits_at_k(df, score_b, k, seed),
            "range": (int(np.quantile(d, a)), int(np.quantile(d, 1 - a))),
            "p_a_better": round(float((d > 0).mean()), 3)}

print("Exploit then EPSS vs. EPSS:", compare(val, ["any_exploit", "epss"], "epss"))
print("EPSS vs. CVSS:             ", compare(val, "epss", "cvss3_score"))

Exploit then EPSS vs. EPSS: {'diff': 6, 'range': (3, 11), 'p_a_better': 0.998}


EPSS vs. CVSS:              {'diff': 3, 'range': (-2, 8), 'p_a_better': 0.794}


**Reading this:** adding exploit code flags in front of EPSS is a real gain: +6 hits, with a range of +3 to +11 that excludes 0. EPSS vs. CVSS is +3, but its range (-2 to +8) includes 0, so on this data we can't claim EPSS beats CVSS. That's the bar for any claim in the write-up: "beats the baseline" only when the paired range excludes 0.

## 6. How much do ties matter?

The tie-break order is random, so a different `seed` can change the score. This matters most for CVSS, where thousands of rows share a 9.8. Here's each baseline over 50 tie-break seeds.

In [12]:
seeds = range(50)
pd.DataFrame([{"ranking": n,
               **pd.Series([hits_at_k(val, s, seed=i) for i in seeds])
                   .describe()[["min", "50%", "max"]].rename({"50%": "median"}).astype(int)}
              for n, s in BASELINES.items()])

,ranking,min,median,max
0,CVSS first,5,5,6
1,EPSS first,8,8,8
2,Exploit code then EPSS,14,14,14


CVSS moves between 5 and 6 here, depending only on which ties land inside the cut. Over 200 seeds it reaches 7 twice (see `phase3_validation.ipynb`). EPSS and exploit-then-EPSS don't move at all, because their scores rarely tie. A model's score will behave like EPSS. The practical point: a one-hit gap between CVSS-based rankings means nothing. **Keep `seed=0` everywhere** so every teammate's numbers are comparable.

## 7. Save the sets for later phases

Phase 4 (features), Phase 5 (models) and Track B (Granite text) all start from these same rows. Saving them avoids rebuilding and guarantees everyone uses identical sets. They go into `data/`, which git ignores.

In [13]:
out = DATA / "backtest"
out.mkdir(exist_ok=True)
val.to_parquet(out / "val_2025.parquet", index=False)
train.to_parquet(out / "train_2024.parquet", index=False)
print("Saved:", *sorted(p.name for p in out.iterdir()))

Saved: train_2024.parquet val_2025.parquet


## Summary

- **The backtest is fair.** The 2025 stand-in has 35,281 rows and 52 positives, and the same baseline wins as on the official benchmark (exploit code then EPSS: 14, vs. CVSS 5 and EPSS 8, which are within noise of each other). `phase3_validation.ipynb` re-checks all of this independently.
- **The bar is 14 hits in the top 243**, and a gain counts only when the paired bootstrap range excludes 0.
- **Noise is large.** A single ranking's 90% range spans about ±6 hits, so decide on the final model early and prefer simpler models when scores are close.
- **Data quirks for Phase 4:** January and February 2025 have low EPSS coverage (55% and 77%), so add a "signal missing" flag. `months_left` ranges from 3 to 11.

**Next:** Phase 4 adds feature groups one at a time and scores each with `hits_at_k` and `compare` against the exploit-then-EPSS baseline.